In [ ]:
import os
from urllib.parse import parse_qs

query_string = os.environ.get("QUERY_STRING", "")
params = parse_qs(query_string)

# Extract values
tenantId = params.get("tenantId", [None])[0]
entityId = params.get("entityId", [None])[0]
filter = None
granularity = params.get("granularity", ["month"])[0]
if granularity == "day":
    filter = "day"
elif granularity == "month":
    filter = "month"
elif granularity == "year":
    filter = "year"
else:
    filter = None  # All time

# Backend config
backendUrl = os.getenv("CASE_MGMT_BACKEND_URL", "http://localhost:3090")
# Service token for authenticated API requests (injected by Voila proxy)
SERVICE_TOKEN = os.getenv('HTTP_X_SERVICE_TOKEN') or os.getenv('SERVICE_TOKEN')
if SERVICE_TOKEN:
    headers = {'Authorization': f'Bearer {SERVICE_TOKEN}'}
else:
    headers = {}
    print("WARNING: No SERVICE_TOKEN found - API calls may fail")

# Other parameters
# dateRange = "all"
page = 1
limit = 1000

# Validation
if not tenantId:
    from IPython.display import display, HTML
    display(HTML("""
        <div style="padding:40px;text-align:center;color:#ef4444;">
            <h3>Missing Required Parameter</h3>
            <p>tenantId is required.</p>
            <code>/voila/render/alert-history.ipynb?endToEndId=xxx&tenantId=yyy</code>
        </div>
    """))
    raise ValueError("tenantId parameter is required")


In [ ]:
try:
    import requests
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import plotly.io as pio
    from plotly.subplots import make_subplots
    from IPython.display import display, HTML
    
    # Set default renderer to basic html/notebook
    pio.renderers.default = "notebook"

    # Output styling
    display(HTML("""
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
        .metric-card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); text-align: center; }
        .metric-value { font-size: 24px; font-weight: bold; color: #111827; }
        .metric-label { font-size: 12px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.05em; }
        .grid-container { display: grid; grid-template-columns: repeat(5, 1fr); gap: 15px; margin-bottom: 20px; }
        .table-container { margin-top: 20px; overflow-x: auto; }
        table { width: 100%; border-collapse: collapse; font-size: 14px; }
        th { text-align: left; padding: 12px; border-bottom: 1px solid #e5e7eb; color: #6B7280; font-weight: 600; }
        td { padding: 12px; border-bottom: 1px solid #e5e7eb; color: #111827; }
        tr:last-child td { border-bottom: none; }
    </style>
    """))
except Exception as e:
    print(f"Import Error: {e}")

In [ ]:
def fetch_json(endpoint: str, params: dict):
    url = f"{backendUrl}{endpoint}"
    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json()

summary = {}
timeline = {}
alerts_resp = {}
df_counts = pd.DataFrame()
df_value = pd.DataFrame()
df_alerts = pd.DataFrame()

try:
    base_params = {
        'tenantId': tenantId,
        'entityId': entityId
    }

    summary = fetch_json(
        "/api/v1/jupyter/proxy/alert-history/summary",
        {**base_params, 'granularity': granularity},
    )

    timeline = fetch_json(
        "/api/v1/jupyter/proxy/alert-history/timeline",
        {**base_params, 'granularity': filter},
    )

    alerts_resp = fetch_json(
        "/api/v1/jupyter/proxy/alert-history/alerts",
        {**base_params, 'granularity': granularity, 'page': page, 'limit': limit},
    )

    # DataFrames
    alert_count = timeline.get('alertCountOverTime', [])
    alert_value = timeline.get('alertValueOverTime', [])
    alerts = alerts_resp.get('alerts', [])

    df_counts = pd.DataFrame(alert_count)
    if not df_counts.empty:
        df_counts['date'] = pd.to_datetime(df_counts['date'])

    df_value = pd.DataFrame(alert_value)
    if not df_value.empty:
        df_value['date'] = pd.to_datetime(df_value['date'])

    df_alerts = pd.DataFrame(alerts)
    if not df_alerts.empty:
        df_alerts['date'] = pd.to_datetime(df_alerts['date'])

except Exception as e:
    display(HTML(f"<div style='color: red; padding: 20px; border: 1px solid red; border-radius: 5px;'>Error fetching data: {str(e)}</div>"))
    print(f"Backend URL: {backendUrl}")

In [ ]:
if summary:
    total_alerts = f"{summary.get('totalAlerts', 0):,}"
    cases_opened = f"{summary.get('casesOpened', 0):,}"
    investigations = f"{summary.get('investigations', 0):,}"
    sar_filings = f"{summary.get('sarFilings', 0):,}"
    total_value = f"${summary.get('totalValue', 0):,.2f}"

    display(HTML(f"""
    <div class="grid-container">
        <div class="metric-card">
            <div class="metric-label">Total Alerts</div>
            <div class="metric-value">{total_alerts}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Cases Opened</div>
            <div class="metric-value">{cases_opened}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Investigations</div>
            <div class="metric-value">{investigations}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">SAR/STR Filings</div>
            <div class="metric-value">{sar_filings}</div>
        </div>
        <div class="metric-card">
            <div class="metric-label">Total Value</div>
            <div class="metric-value">{total_value}</div>
        </div>
    </div>
    """))

In [ ]:
if (not df_counts.empty) or (not df_value.empty):
    from datetime import timedelta
    import calendar
    
    # Determine the date range from the data
    if not df_counts.empty:
        min_date = df_counts['date'].min()
        max_date = df_counts['date'].max()
    elif not df_value.empty:
        min_date = df_value['date'].min()
        max_date = df_value['date'].max()
    
    # Get the first and last day of the month
    first_day = min_date.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    last_day_num = calendar.monthrange(min_date.year, min_date.month)[1]
    last_day = min_date.replace(day=last_day_num, hour=23, minute=59, second=59, microsecond=999999)
    
    # Data normalization: Generate complete date range and fill missing dates with zeros
    date_range = pd.date_range(start=first_day, end=last_day, freq='D')
    df_complete_dates = pd.DataFrame({'date': date_range})
    
    # Normalize Alert Count data
    if not df_counts.empty:
        df_counts_daily = df_counts.copy()
        df_counts_daily['date'] = df_counts_daily['date'].dt.normalize()
        
        # Merge with complete date range
        df_counts_normalized = df_complete_dates.merge(
            df_counts_daily[['date', 'alerts', 'cases', 'investigations']], 
            on='date', 
            how='left'
        )
        
        # Fill missing values with 0
        df_counts_normalized['alerts'] = df_counts_normalized['alerts'].fillna(0).astype(int)
        df_counts_normalized['cases'] = df_counts_normalized['cases'].fillna(0).astype(int)
        df_counts_normalized['investigations'] = df_counts_normalized['investigations'].fillna(0).astype(int)
    else:
        df_counts_normalized = df_complete_dates.copy()
        df_counts_normalized['alerts'] = 0
        df_counts_normalized['cases'] = 0
        df_counts_normalized['investigations'] = 0
    
    # Normalize Alert Value data
    if not df_value.empty:
        df_value_daily = df_value.copy()
        df_value_daily['date'] = df_value_daily['date'].dt.normalize()
        
        df_value_normalized = df_complete_dates.merge(
            df_value_daily[['date', 'totalValue']],
            on='date',
            how='left'
        )
        
        df_value_normalized['totalValue'] = df_value_normalized['totalValue'].fillna(0)
    else:
        df_value_normalized = df_complete_dates.copy()
        df_value_normalized['totalValue'] = 0
    
    # Calculate X-axis range with padding
    x_range_start = (first_day - timedelta(days=0.5)).strftime('%Y-%m-%d')
    x_range_end = (last_day + timedelta(days=0.5)).strftime('%Y-%m-%d')
    
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.15,
        subplot_titles=(
            "Alert Count Over Time",
            "Alert Value Over Time",
        ),
        row_heights=[0.5, 0.5],
    )

    # Alert Count Over Time - Multi-line time series with normalized data
    fig.add_trace(
        go.Scatter(
            x=df_counts_normalized['date'],
            y=df_counts_normalized['alerts'],
            mode='lines+markers',
            name='Alerts',
            line=dict(color='#ef4444', width=2.5, shape='spline'),
            marker=dict(size=6, symbol='circle', line=dict(width=1, color='white')),
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Alerts: %{y}<br>" +
                "<extra></extra>"
            ),
            connectgaps=False,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=df_counts_normalized['date'],
            y=df_counts_normalized['cases'],
            mode='lines+markers',
            name='Cases',
            line=dict(color='#3b82f6', width=2.5, shape='spline'),
            marker=dict(size=6, symbol='circle', line=dict(width=1, color='white')),
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Cases: %{y}<br>" +
                "<extra></extra>"
            ),
            connectgaps=False,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=df_counts_normalized['date'],
            y=df_counts_normalized['investigations'],
            mode='lines+markers',
            name='Investigations',
            line=dict(color='#8b5cf6', width=2.5, shape='spline'),
            marker=dict(size=6, symbol='circle', line=dict(width=1, color='white')),
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Investigations: %{y}<br>" +
                "<extra></extra>"
            ),
            connectgaps=False,
        ),
        row=1,
        col=1,
    )

    # Alert Value Over Time - Bar chart with normalized data
    fig.add_trace(
        go.Bar(
            x=df_value_normalized['date'],
            y=df_value_normalized['totalValue'],
            name='Total Alert Value',
            marker_color='#10b981',
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Value: $%{y:,.2f}<br>" +
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

    fig.update_layout(
        height=760,
        showlegend=True,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.08,
            xanchor='center',
            x=0.5,
            bgcolor="rgba(255, 255, 255, 0.8)",
            bordercolor="#E5E7EB",
            borderwidth=1,
            font=dict(size=11)
        ),
        margin=dict(l=60, r=60, t=90, b=40),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        hovermode='x unified',
        font=dict(
            family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif",
            color="#111827"
        ),
    )

    # Grid styling
    fig.update_xaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(229, 231, 235, 0.5)',
        showline=True,
        linewidth=1,
        linecolor='#E5E7EB',
        mirror=False
    )
    fig.update_yaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(229, 231, 235, 0.5)',
        showline=True,
        linewidth=1,
        linecolor='#E5E7EB',
        mirror=False,
        zeroline=True,
        zerolinewidth=1,
        zerolinecolor='#E5E7EB'
    )

    # Row 1: Alert Count - X-axis configuration
    fig.update_xaxes(
        row=1, 
        col=1,
        title_text="<b>Date (Full Month)</b>",
        tickformat='%b %d',
        tickangle=-45,
        tickmode='auto',
        nticks=min(last_day_num, 20),  # Prevent label overlap
        range=[x_range_start, x_range_end],
        tickfont=dict(size=9),
        title_font=dict(size=12, color="#6B7280"),
        fixedrange=False
    )
    fig.update_yaxes(
        row=1, 
        col=1, 
        title_text="<b>Count</b>",
        title_font=dict(size=12, color="#6B7280"),
        rangemode='tozero',
        tick0=0,
        dtick='auto'
    )
    
    # Row 2: Alert Value - X-axis configuration
    fig.update_xaxes(
        row=2, 
        col=1,
        title_text="<b>Date (Full Month)</b>",
        tickformat='%b %d',
        tickangle=-45,
        tickmode='auto',
        nticks=min(last_day_num, 20),
        range=[x_range_start, x_range_end],
        tickfont=dict(size=9),
        title_font=dict(size=12, color="#6B7280"),
        fixedrange=False
    )
    fig.update_yaxes(
        row=2, 
        col=1, 
        title_text="<b>Total Value ($)</b>",
        title_font=dict(size=12, color="#10B981"),
        rangemode='tozero'
    )

    # Add crosshair effect with spike lines
    fig.update_xaxes(showspikes=True, spikemode='across', spikesnap='cursor', spikecolor='#9CA3AF', spikethickness=1)
    fig.update_yaxes(showspikes=True, spikemode='across', spikesnap='cursor', spikecolor='#9CA3AF', spikethickness=1)

    # Show chart with animation
    fig.show()

In [ ]:
if not df_alerts.empty:
    display(HTML("<h3>Alert History</h3>"))

    # Format columns for readability
    display_df = df_alerts.copy()
    if 'date' in display_df.columns:
        display_df['date'] = pd.to_datetime(display_df['date']).dt.strftime('%Y-%m-%d')

    rename_map = {
        'alertId': 'Alert ID',
        'type': 'Type',
        'severity': 'Severity',
        'status': 'Status',
        'caseId': 'Case ID',
        'outcome': 'Outcome',
        'date': 'Date',
    }

    display_df = display_df[[c for c in rename_map.keys() if c in display_df.columns]].rename(columns=rename_map)

    # Render HTML table
    html = display_df.to_html(index=False, classes='table')
    display(HTML(f"<div class='table-container'>{html}</div>"))
else:
    display(HTML("<p>No alerts found for this Transaction.</p>"))